In [6]:
import pandas as pd 
import yaml

config_path = "/data/gpfs-1/users/kisa11_c/work/coding/80K_analysis/global80K_config.yaml"
config_path = "/home/kisa/coding/80K_MPRA/80K-Analysis/global80K_config.yaml"
with open(config_path, 'r') as ymlfile:
    config = yaml.safe_load(ymlfile)

metadata_path = config['files']['creating']['metadata_table']
old_metadata_path = config['files']['creating']['old_metadata_table']

metadata_df = pd.read_csv(old_metadata_path, sep='\t')
metadata_df
metadata_df
# change datatypes:
metadata_df['start'] = metadata_df['start'].astype('Int64').astype(str)
metadata_df['end'] = metadata_df['end'].astype('Int64').astype(str)
# set NaN values
metadata_df['start'] = metadata_df['start'].replace('<NA>', 'NaN')
metadata_df['end'] = metadata_df['end'].replace('<NA>', 'NaN')

In [8]:
metadata_df.head()

,name,sequence,category,class,source,ref,chr,start,end,variant_class,variant_pos,SPDI,allele,info
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTAAGAATACAAGTAACTGATGAATGAAGGGGG...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,NaN,NaN,NaN,NaN,NaN,NaN,NaN,cardiac_neuro_cava_random
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTTTGGGTATGCTGCCCCCCAGCTGGCGGGGCA...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,NaN,NaN,NaN,NaN,NaN,NaN,NaN,cardiac_neuro_cava_random
2,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTACGAGCAAGGGAATGAGAGAGAGTGGGTTAG...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,NaN,NaN,NaN,NaN,NaN,NaN,NaN,cardiac_neuro_cava_random
3,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCGTGGACACGCGTGATTGACCCTTTAACTGT...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,NaN,NaN,NaN,NaN,NaN,NaN,NaN,cardiac_neuro_cava_random
4,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCCGGAGAGTCTCAGCTCCCGCAGCCCTAACA...,element,test,"candidate CRE nearby cardiac, neuro, cava and ...",GRCh38,NaN,NaN,NaN,NaN,NaN,NaN,NaN,cardiac_neuro_cava_random


### Control regions:
- `/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/design/final_design/design.control_sequences.tsv`

In [3]:
region_tbl = '/data/gpfs-1/users/kisa11_c/work/coding/80K_analysis/05_variant_region_list/resources/design.control_regions.tsv'
control_region_tbl = pd.read_csv(region_tbl, sep='\t')
control_region_tbl

,sample,bed_file
0,GC_Cort_Chengyu,/data/gpfs-1/users/kisa11_c/work/coding/MPRAOl...
1,GC_GABA_Chengyu,/data/gpfs-1/users/kisa11_c/work/coding/MPRAOl...
2,GC_Glut_Chengyu,/data/gpfs-1/users/kisa11_c/work/coding/MPRAOl...
3,GC_Hon,/data/gpfs-1/users/kisa11_c/work/coding/MPRAOl...
4,GC_Vista,/data/gpfs-1/users/kisa11_c/work/coding/MPRAOl...
5,GC_DNase_positive,/data/gpfs-1/users/kisa11_c/work/coding/MPRAOl...
6,GC_DNase_negative_brain,/data/gpfs-1/users/kisa11_c/work/coding/MPRAOl...
7,GC_DNase_negative_blood,/data/gpfs-1/users/kisa11_c/work/coding/MPRAOl...


In [9]:
def check_region_length(bed_df, sample, expected_length=270):
    """Check if all regions in a bed file have the expected length"""

    bed_df['length'] = bed_df['end'] - bed_df['start']
    is_twoseventy = bed_df['length'] == 270
    if is_twoseventy.sum() == bed_df.shape[0]:
        print(f'{sample} has only 270bp regions')
    else:
        print(f'{sample} has regions of different lengths')

# iterate over table and read bed files
for index, row in control_region_tbl.iterrows():
    # sample:
    sample = row['sample']
    bed_file = row['bed_file']
    # read bed file
    bed_df = pd.read_csv(bed_file, sep='\t', header=None)
    bed_df.columns = ['chr', 'start', 'end', 'id', 'score', 'strand']
    # check region length:
    check_region_length(bed_df, sample, 270)
    print(bed_file)
    print(bed_df['id'].str.contains('~').sum())
    print(bed_df.shape[0])
    break
    
    

GC_Cort_Chengyu has only 270bp regions
/data/gpfs-1/users/kisa11_c/work/coding/MPRAOligoDesign/resources/controls/Cort/primary_fetal_cortical_cells-active-hg38.bed.gz
0
195


### Numbers of controls
- check number of controls with variants 
  - started in varinat_region_list (REF ALT map for variants)
  - problem with ref alt of variants: not all variants which seem to have an alternative sequence have such a corresponding sequence => Which variant controls can be used? And how many are this?
- check positive control and negative controls and neutral controls

In [12]:
def get_label(header):
    """Return label from header <label>:xxx"""
    return header.split(":")[0]

In [15]:
pre_control = metadata_df[~metadata_df['name'].str.startswith('cardiac_neuro_cava_random')]
pre_control.head()
pre_control['tmp_label'] = pre_control['name'].apply(get_label)

pre_control

/tmp/ipykernel_22005/1372476666.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pre_control['tmp_label'] = pre_control['name'].apply(get_label)


,name,sequence,category,class,source,ref,chr,start,end,variant_class,variant_pos,SPDI,allele,info,tmp_label
73940,GC_Atrial_fib:rs7795510|CAV1|STARR-seq-AF~rs78...,AGGACCGGATCAACTTCATTTCATTATAATCAAAAAGGATTTTTAA...,element,element inactive control,IGVF general controls (GC_Atrial_fib),GRCh38,NaN,NaN,NaN,NaN,NaN,NaN,NaN,GC_Atrial_fib,GC_Atrial_fib
73941,GC_Atrial_fib:REF_rs74541936|KCNN3|STARR-seq-A...,AGGACCGGATCAACTCAGCTGCCCATGCTGGGACTGTGATTTTTTG...,variant,variant negative control,IGVF general controls (GC_Atrial_fib),GRCh38,NaN,NaN,NaN,SNP,NaN,NaN,ref,GC_Atrial_fib,GC_Atrial_fib
73942,GC_Atrial_fib:REF_rs34292822|KCNN3|STARR-seq-A...,AGGACCGGATCAACTAGAGCCCTTCTGGGGGCCCTGGCCACTGGCC...,variant,variant negative control,IGVF general controls (GC_Atrial_fib),GRCh38,NaN,NaN,NaN,SNP,NaN,NaN,ref,GC_Atrial_fib,GC_Atrial_fib
73943,GC_Atrial_fib:REF_rs12754189|KCNN3|STARR-seq-A...,AGGACCGGATCAACTAGGAAAGGCACTGGAAATTGTACTTACTCCA...,variant,variant negative control,IGVF general controls (GC_Atrial_fib),GRCh38,NaN,NaN,NaN,SNP,NaN,NaN,ref,GC_Atrial_fib,GC_Atrial_fib
73944,GC_Atrial_fib:REF_rs36088503|KCNN3|STARR-seq-A...,AGGACCGGATCAACTTTTGCAAAGGTATGGTTGGTGGATGGAGAAA...,variant,variant negative control,IGVF general controls (GC_Atrial_fib),GRCh38,NaN,NaN,NaN,SNP,NaN,NaN,ref,GC_Atrial_fib,GC_Atrial_fib
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
80210,MK:tile_2240|chr1-116244322+116244591|scramble...,AGGACCGGATCAACTCTTAATCAAATAACCCATTAATTCTATATAT...,scrambled,element inactive control,NaN,GRCh38,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MK,MK
80211,MK:tile_6675|chr11-2374617+2374886|scramble_ne...,AGGACCGGATCAACTCATCGGCCCTGGTGAAGCGTCCGTCCAGACG...,scrambled,element inactive control,NaN,GRCh38,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MK,MK
80212,MK:tile_18415|chr17-71181691+71181960|scramble...,AGGACCGGATCAACTTAAATATTCAGCGATACATTCCTATTCTTTT...,scrambled,element inactive control,NaN,GRCh38,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MK,MK
80213,MK:tile_14356|chr15-67031618+67031887|scramble...,AGGACCGGATCAACTTGAAGCCCCTGATTCTGTTAGAATAAGGTTA...,scrambled,element inactive control,NaN,GRCh38,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MK,MK
